# RXN2: prepare Lowe USPTO reactions (weak pretraining only)

This notebook streams the verified CC0 Lowe USPTO archives already in Drive into a JSONL file for **weak reaction pretraining**. It does not create accepted process steps, routes, scale labels, or gold training examples.

Run the cells from top to bottom in Google Colab. No USPTO account or API key is required.

## 1. Mount Drive and define the pinned paths

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/RXN2')
SNAPSHOT = DRIVE_ROOT / 'data/raw/lowe_uspto_reactions/2017-06-13'
OUTPUT = DRIVE_ROOT / 'data/processed/lowe_uspto_reactions/2017-06-13/weak-reactions.jsonl'
TOOLS_ROOT = DRIVE_ROOT / 'colab-tools/lowe-uspto-2017-06-13'
PREPARE_SCRIPT = TOOLS_ROOT / 'scripts/prepare_lowe_uspto.py'

assert SNAPSHOT.is_dir(), f'Missing snapshot: {SNAPSHOT}'
assert (SNAPSHOT / 'release-metadata.json').is_file(), 'Missing release metadata'
assert PREPARE_SCRIPT.is_file(), f'Missing staged RXN2 tool: {PREPARE_SCRIPT}'
print('Snapshot:', SNAPSHOT)
print('Output:', OUTPUT)

## 2. Install the archive reader

In [ ]:
!apt-get -qq update && apt-get -qq install -y p7zip-full

## 3. Stream the archives into weak-training JSONL

This validates the pinned source checksums before writing. It can take several minutes; do not interrupt it. The final file is written atomically, so a partial file is never treated as complete.

In [ ]:
import subprocess
import sys

command = [
    sys.executable, str(PREPARE_SCRIPT),
    '--snapshot', str(SNAPSHOT),
    '--output', str(OUTPUT),
    '--seven-zip', '7z',
]
print(' '.join(command))
subprocess.run(command, check=True)

## 4. Validate the output boundary

In [ ]:
import itertools
import json

MANIFEST = OUTPUT.with_name('manifest.json')
manifest = json.loads(MANIFEST.read_text(encoding='utf-8'))
assert manifest['complete'] is True
assert manifest['supervision_tier'] == 'weak_pretraining'
assert manifest['automatic_acceptance'] is False
assert OUTPUT.is_file() and OUTPUT.stat().st_size > 0

with OUTPUT.open(encoding='utf-8') as handle:
    preview = [json.loads(line) for line in itertools.islice(handle, 3)]

print(json.dumps({
    'records': manifest['records'],
    'output_sha256': manifest['output_sha256'],
    'supervision_tier': manifest['supervision_tier'],
}, indent=2))
preview

## Next

Leave this output separate from the verified patent-route dataset. Next, return to the RXN2 review queue and accept or reject evidence-backed process steps and routes; only those accepted records may become gold training data.